# R2-05 · Modelos basados en árboles — Profundización (opcional) 🔬

**Formación Pública — Rama R2 · Científico de Datos · Notebook de profundización**

Opcional y de **transferencia**: árboles, sobreajuste e importancia de variables sobre otro dato
público, el catálogo de **sismos M ≥ 4,5 en Chile 2020–2025** del Servicio Geológico de EE.UU. (USGS).
¿Se puede predecir la **profundidad** de un sismo con su ubicación y magnitud? Un árbol de 3 preguntas
te va a mostrar algo que los geólogos ya saben.

> Cada ejercicio termina en una **celda de chequeo** con ✅ / pista.

In [ ]:
import pandas as pd, numpy as np, os, urllib.request
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
CSV = "sismos.csv"
if not os.path.exists(CSV):
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/{CSV}", CSV)
sis = pd.read_csv(CSV)    # sismos M ≥ 4,5 en Chile 2020–2025 (USGS)
print("sismos:", sis.shape)
sis.head()

## 1. ¿A qué profundidad ocurre un sismo?

Cada fila es un sismo con su ubicación, magnitud y **profundidad** (km). ¿Se puede predecir la profundidad solo con latitud, longitud y magnitud? Aquí no pronosticamos el futuro sino una relación física, así que un **corte aleatorio** es razonable (con una advertencia: las réplicas de un mismo terremoto se parecen entre sí). Primero, el baseline: predecir siempre la profundidad promedio.

### ✍️ Ejercicio 1 — Separa 70/30 y mide el baseline

In [ ]:
FEAT = ["latitud", "longitud", "magnitud"]
X = sis[FEAT]
y = sis["profundidad_km"]
# TODO: train_test_split(X, y, test_size=0.3, random_state=42)
X_train, X_test, y_train, y_test = ..., ..., ..., ...
# TODO: MAE de predecir siempre y_train.mean()
mae_base = ...
print("MAE baseline:", mae_base)

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert len(X_train) + len(X_test) == len(sis)
    assert abs(len(X_test) / len(sis) - 0.3) < 0.01
    assert abs(mae_base - mean_absolute_error(y_test, np.full(len(y_test), y_train.mean()))) < 1e-9
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'train_test_split(X, y, test_size=0.3, random_state=42); np.full(len(y_test), y_train.mean()).')
    print("   Detalle:", e)

## 2. Un árbol que se puede leer

Entrena un árbol de profundidad 3 e **imprime sus reglas** con `export_text`. Fíjate en qué variable pregunta primero.

### ✍️ Ejercicio 2 — Árbol de profundidad 3: MAE y reglas

In [ ]:
# TODO: árbol de profundidad 3 (random_state=42) entrenado con (X_train, y_train)
arbol = ...
# TODO: MAE en prueba
mae_arbol = ...
print("MAE árbol:", mae_arbol, "| baseline:", mae_base)
print(export_text(arbol, feature_names=FEAT))

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    assert arbol.get_depth() == 3
    assert mae_arbol < mae_base / 2, "El árbol debería reducir el error a menos de la mitad"
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'DecisionTreeRegressor(max_depth=3, random_state=42).fit(X_train, y_train).')
    print("   Detalle:", e)

## 3. Sobreajuste, otra vez

Sin límite de profundidad, el árbol memoriza los 1.120 sismos de entrenamiento. Compruébalo.

### ✍️ Ejercicio 3 — Árbol sin límite: entrenamiento vs prueba

In [ ]:
# TODO: árbol sin max_depth (random_state=42) y sus MAE de entrenamiento y prueba
arbol_profundo = ...
mae_train = ...
mae_test = ...
print("MAE entrenamiento:", mae_train, "| MAE prueba:", mae_test)

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    assert mae_train < 1, "Sin límite, el error de entrenamiento es casi 0"
    assert mae_test > mae_train + 5, "…pero en prueba el error es mucho mayor"
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'DecisionTreeRegressor(random_state=42).fit(X_train, y_train); MAE con predict(X_train) y predict(X_test).')
    print("   Detalle:", e)

## 4. El bosque y la física: ¿qué variable manda?

Entrena un bosque aleatorio y mira la importancia de variables. Si la geología tiene razón, una de las tres debería dominar.

### ✍️ Ejercicio 4 — Bosque aleatorio: MAE e importancia

In [ ]:
# TODO: RandomForestRegressor(n_estimators=100, min_samples_leaf=5, random_state=42), entrenado
bosque = ...
mae_bosque = mean_absolute_error(y_test, bosque.predict(X_test))
importancias = pd.Series(bosque.feature_importances_, index=FEAT).sort_values(ascending=False)
# TODO: la feature más importante
feature_top = ...
print("MAE bosque:", mae_bosque, "| feature más importante:", feature_top)

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    assert mae_bosque < mae_arbol
    assert feature_top == "longitud"
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'bosque.fit(X_train, y_train) y importancias.idxmax().')
    print("   Detalle:", e)

## La subducción, descubierta por un árbol

La **longitud** manda: mientras más al **este** (tierra adentro) ocurre un sismo, más **profundo**
es. Es la huella de la **subducción**: la placa de Nazca se hunde bajo la Sudamericana, inclinada
hacia el este, y los sismos siguen esa placa. El árbol no sabe geología — la encontró en los datos.
La magnitud casi no aporta: un sismo grande puede ser superficial o profundo.

In [ ]:
# (ilustración) Profundidad según longitud: la placa que se hunde hacia el este
try:
    plt.figure(figsize=(7, 4))
    plt.scatter(sis["longitud"], sis["profundidad_km"], s=8, alpha=0.4, color="#4240e5")
    plt.gca().invert_yaxis()
    plt.xlabel("longitud (° — más a la derecha = más al este)"); plt.ylabel("profundidad (km)")
    plt.title("Sismos M ≥ 4,5 en Chile 2020–2025"); plt.tight_layout(); plt.show()
except Exception:
    print("ℹ️ Ejecuta la celda de preparación primero.")

---

*Fin de la profundización de R2-05.*